In [ ]:
rundir = '/gpfs/commons/home/tbotella/bam-readcount-rs/bench/results/latest'

In [ ]:
from pathlib import Path
import polars as pl
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'svg'
mpl.rcParams['figure.dpi'] = 110
mpl.rcParams['savefig.dpi'] = 200
mpl.rcParams['savefig.bbox'] = 'tight'
mpl.rcParams['font.family'] = 'DejaVu Sans'
rundir = Path(rundir)
plots = rundir / 'plots'
plots.mkdir(parents=True, exist_ok=True)
joined_dir = rundir / 'joined'
print('rundir:', rundir, '\nplots:', plots, '\nparquets:', sorted(joined_dir.iterdir())[:3])

In [ ]:
INFO = [
    'count','avg_mapping_quality','avg_basequality','avg_se_mapping_quality',
    'num_plus_strand','num_minus_strand','avg_pos_as_fraction',
    'avg_num_mismatches_as_fraction','avg_sum_mismatch_qualities',
    'num_q2_containing_reads','avg_distance_to_q2_start_in_q2_reads',
    'avg_clipped_length','avg_distance_to_effective_3p_end',
]
lf = pl.scan_parquet(str(joined_dir / '*.parquet'))
n_rows = lf.select(pl.len()).collect().item()
n_samples = sum(1 for _ in joined_dir.glob('*.parquet'))
print(f'Lazy scan over {n_samples} samples, {n_rows:,} joined rows.')

In [ ]:
# Per-feature correlation, computed in polars over the lazy scan (no full
# materialization). For each metric: pearson r, MAE, %exact.
rows = []
for m in INFO:
    a_col, b_col = f'ref_{m}', f'rs_{m}'
    stats = (
        lf.select([
            pl.corr(pl.col(a_col), pl.col(b_col)).alias('pearson_r'),
            (pl.col(a_col) - pl.col(b_col)).abs().mean().alias('mae'),
            ((pl.col(a_col) - pl.col(b_col)).abs() <= 0.01).mean().alias('frac_within_001'),
            (pl.col(a_col) == pl.col(b_col)).mean().alias('frac_exact'),
            pl.len().alias('n'),
        ]).collect()
    )
    rows.append({
        'metric': m,
        'n': int(stats['n'][0]),
        'pearson_r': float(stats['pearson_r'][0] or 1.0),
        'mae': float(stats['mae'][0] or 0.0),
        'pct_within_001': float(stats['frac_within_001'][0] * 100.0),
        'pct_exact': float(stats['frac_exact'][0] * 100.0),
    })
corr_df = pl.DataFrame(rows)
corr_df.write_csv(rundir / 'per_feature_corr.tsv', separator='\t')
print(corr_df)

In [ ]:
# CORRELATION GRID — sample 30k points per metric to keep plot tractable.
import math
ncols = 4
nrows = math.ceil(len(INFO) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.0*ncols, 3.6*nrows))
axes = axes.flatten()
panel_letters = list('ABCDEFGHIJKLMN')
rng = np.random.default_rng(0)
for i, m in enumerate(INFO):
    ax = axes[i]
    sample = (
        lf.select([pl.col(f'ref_{m}'), pl.col(f'rs_{m}')])
          .filter(pl.col(f'ref_{m}').is_finite() & pl.col(f'rs_{m}').is_finite())
          .collect(streaming=True)
    )
    a = sample[f'ref_{m}'].to_numpy()
    b = sample[f'rs_{m}'].to_numpy()
    if len(a) > 30000:
        idx = rng.choice(len(a), size=30000, replace=False)
        a, b = a[idx], b[idx]
    r = corr_df.filter(pl.col('metric') == m)['pearson_r'][0]
    pct = corr_df.filter(pl.col('metric') == m)['pct_exact'][0]
    ax.scatter(a, b, s=1.5, alpha=0.18, edgecolor='none', color='#3367d6')
    if len(a) > 0:
        lo = float(min(a.min(), b.min()))
        hi = float(max(a.max(), b.max()))
        ax.plot([lo, hi], [lo, hi], color='#bbbbbb', lw=0.8, ls='--', zorder=0)
    ax.set_xlabel(f'upstream {m}', fontsize=8)
    ax.set_ylabel(f'rs {m}', fontsize=8)
    ax.tick_params(labelsize=7)
    color = '#0a8a3a' if r >= 0.99 else ('#c4a000' if r >= 0.95 else '#a30000')
    ax.set_title(f'{panel_letters[i]}  {m}\nr={r:.4f}  exact={pct:.1f}%', fontsize=8.5, color=color)
for j in range(len(INFO), len(axes)):
    axes[j].axis('off')
fig.suptitle(f'bam-readcount-rs vs upstream — {n_rows:,} (sample,position,base) rows across {n_samples} samples', fontsize=11, y=1.005)
fig.tight_layout()
fig.savefig(plots / 'correlation_grid.png')
plt.show()

In [ ]:
# RUNTIME / MEMORY (bam-readcount-rs only — upstream timings would come
# from Nextflow trace files, see README).
ms = pl.read_csv(rundir / 'per_sample_metrics.tsv', separator='\t').with_columns([
    pl.col('rs_wall_s').cast(pl.Float64),
    pl.col('rs_peak_rss_kb').cast(pl.Float64),
    pl.col('bed_lines').cast(pl.Int64),
])
print(ms.describe())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
for c in ms['cohort'].unique().to_list():
    sub = ms.filter(pl.col('cohort') == c)
    ax.scatter(sub['bed_lines'], sub['rs_wall_s'], label=c, s=14, alpha=0.7)
ax.set_xlabel('queried sites (BED lines)')
ax.set_ylabel('wall time (s, 4 threads)')
ax.set_title(f'Runtime — median {ms["rs_wall_s"].median():.1f}s across {len(ms)} samples')
ax.legend(fontsize=8, frameon=False)
ax.set_xscale('log'); ax.set_yscale('log')
ax = axes[1]
for c in ms['cohort'].unique().to_list():
    sub = ms.filter(pl.col('cohort') == c)
    ax.scatter(sub['bed_lines'], sub['rs_peak_rss_kb'] / 1024.0, label=c, s=14, alpha=0.7)
ax.set_xlabel('queried sites (BED lines)')
ax.set_ylabel('peak RSS (MB)')
ax.set_title(f'Memory — median {ms["rs_peak_rss_kb"].median()/1024:.0f} MB')
ax.legend(fontsize=8, frameon=False)
ax.set_xscale('log')
fig.tight_layout()
fig.savefig(plots / 'runtime_memory.png')
plt.show()

In [ ]:
# SUMMARY.md — picked up by README
lines = [
    f'# Benchmark summary — `{rundir.name}`',
    '',
    f'- samples scored: **{n_samples}**',
    f'- joined rows (per-sample × position × base): **{n_rows:,}**',
    f'- median rs wall time: **{ms["rs_wall_s"].median():.1f}s** (4 threads)',
    f'- median rs peak RSS: **{ms["rs_peak_rss_kb"].median()/1024:.0f} MB**',
    '',
    '## Per-feature correlation',
    '',
    '| metric | n | pearson r | MAE | % exact | % within 0.01 |',
    '|---|---:|---:|---:|---:|---:|',
]
for r in corr_df.iter_rows(named=True):
    lines.append(f"| {r['metric']} | {r['n']:,} | {r['pearson_r']:.5f} | {r['mae']:.4g} | {r['pct_exact']:.2f}% | {r['pct_within_001']:.2f}% |")
(rundir / 'SUMMARY.md').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))